# Florence-2-large — DIMER multi-capability vision-language tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/florence2-vision-language-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/florence2-vision-language-pipeline/blob/main/tutorials/florence2_vision_language_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-florence--community%2FFlorence--2--large-ffcc4d?style=flat)](https://huggingface.co/florence-community/Florence-2-large)
[![Upstream](https://img.shields.io/badge/Upstream-microsoft%2FFlorence--2--large-181717?style=flat&logo=huggingface&logoColor=white)](https://huggingface.co/microsoft/Florence-2-large)
[![arXiv](https://img.shields.io/badge/arXiv-2311.06242-b31b1b.svg)](https://arxiv.org/abs/2311.06242)

**Profile:** `MULTI-CAPABILITY`  
**Notebook specification:** DIMER Notebook Specification 1.0  
**Capability:** prompt-selected vision-language tasks — image captioning, object detection and OCR demonstrated — using the pinned Florence-2-large weights

This notebook is the executable reference path for the repository capability. It exercises the repository's public pipeline API (`Florence2Pipeline`) rather than reimplementing model inference. Florence-2 is one sequence-to-sequence model whose behaviour is selected by a **task prompt token**: the image is resized to 768 x 768 by the processor (aspect ratio is not preserved), encoded into 577 visual tokens, and the decoder generates text that the processor then parses per task — plain text for captions and OCR, boxes plus labels in input-pixel coordinates for region tasks. Decoding is **deterministic beam search** (`num_beams` = 3 from the snapshot generation config, `do_sample=False`), so a rerun on the same device, dtype and library versions reproduces the same output. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting — the pinned checkpoint is used as published. The pin is the community "official transformers converted checkpoint" (`florence-community/Florence-2-large`), loadable by native `transformers` classes with `trust_remote_code=False`; the original `microsoft/Florence-2-large` snapshot needs remote code and was rejected (see `../docs/WEIGHTS.md`). What upstream supplies is the model, processor and task-token convention; what this repository adds is manifest verification, input validation and ceilings, a fixed per-task output contract, and the `character_error_rate` helper.

**Learning objectives:** bootstrap the repository in a fresh runtime, generate a synthetic image with drawn shapes and drawn text (or upload your own), surface the pipeline's ceilings and the task list, stage and digest-verify the immutable upstream snapshot, run three capabilities — `<CAPTION>`, `<OD>` and `<OCR>` — through the public API with explicit generation settings, read each capability's output contract correctly, understand which capability has a metric (OCR against known text, sanity only) and which have none, and export machine-readable results plus provenance.

**Exposed but not demonstrated:** the other task tokens in `TASKS` — `<DETAILED_CAPTION>`, `<MORE_DETAILED_CAPTION>`, `<DENSE_REGION_CAPTION>`, `<REGION_PROPOSAL>`, `<OCR_WITH_REGION>` and `<CAPTION_TO_PHRASE_GROUNDING>` (the only task that takes a `text_input`). **This notebook and this repository do not provide:** segmentation of any kind (`<REFERRING_EXPRESSION_SEGMENTATION>`, `<REGION_TO_SEGMENTATION>`), region-to-category or region-to-description prompts, open-vocabulary detection, visual question answering, batched inference, confidence scores for boxes or text, or fine-tuning. The pipeline rejects any task token outside `TASKS`.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available (float16 there, the dtype the checkpoint ships in); on the model card's CPU smoke the snapshot loaded in 7.1 s, `<CAPTION>` took 4.9 s and `<OD>` 9.0 s on a 256 x 256 drawing, so the three-task default runs in about a minute on a hosted CPU runtime. The pinned `torch==2.14.0` install and the 1.55 GB checkpoint are the largest downloads of the run.
- **Knowledge:** basic Python and PIL image handling; what beam search is; what a bounding box in pixel coordinates is; what a character error rate measures.
- **Data:** the default sample is a synthetic image generated in code; BYOD is one image file, gated off by default. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded images remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Git clone, the pinned wheel installs, and the fetch of the missing snapshot file from the Hugging Face Hub at the immutable revision. No credentials are needed.

## 1. Bootstrap the repository and pinned runtime

When the notebook is opened without a repository checkout, this cell clones the repository. Released notebooks default to `main`; automated candidate validation can set `DIMER_TUTORIAL_REF` to an immutable commit or review branch. The repository is installed as a regular (non-editable) package so it is importable in this same runtime; an editable install would only become importable after a restart. Model-facing dependencies (`torch`, `transformers`, `huggingface-hub`, `safetensors`, `numpy`, `pillow`) are directly pinned by `pyproject.toml`. If installation replaces any package that this runtime has already imported (hosted runtimes commonly pre-import a different NumPy or Pillow), the cell fails with a restart instruction rather than continuing with mixed versions: restart the runtime and rerun from the top. Inference runs in float32 on CPU and float16 on CUDA (the checkpoint's shipped dtype), so generated text can differ between the two device paths; no compilation or quantization is applied.

In [ ]:
import importlib
import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/kurtvalcorza/florence2-vision-language-pipeline.git'
REPO_NAME = 'florence2-vision-language-pipeline'
REPO_REF = os.environ.get('DIMER_TUTORIAL_REF', 'main')
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    checkout = ROOT / REPO_NAME
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '-q', REPO_URL, str(checkout)], check=True)
    if REPO_REF != 'main':
        subprocess.run(['git', '-C', str(checkout), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', '-C', str(checkout), 'checkout', '-q', 'main'], check=True)
        subprocess.run(['git', '-C', str(checkout), 'pull', '--ff-only', '-q', 'origin', 'main'], check=True)
    os.chdir(checkout)
    ROOT = Path.cwd()

if not SKIP_INSTALL:
    # Every distribution that is already imported in this runtime is captured before installation,
    # whatever its name (PIL -> pillow), so a pinned install that replaces any loaded package is
    # detected. Distribution metadata is compared with metadata afterwards: torch.__version__ carries
    # a local build label (for example 2.14.0+cu130) that the distribution version omits.
    def _installed_version(distribution):
        try:
            return importlib.metadata.version(distribution)
        except importlib.metadata.PackageNotFoundError:
            return None
    _module_dists = importlib.metadata.packages_distributions()
    _loaded_dists = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded_dists}
    # Non-editable install: an editable (.pth) install is not importable until the
    # interpreter restarts, which a fresh hosted runtime cannot do mid-notebook.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(ROOT)], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

REPO_SHA = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
import platform, PIL, numpy, torch, transformers
print({'repository': str(ROOT), 'repository_revision': REPO_SHA, 'requested_ref': REPO_REF, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'pillow': PIL.__version__, 'cuda': torch.cuda.is_available()})

## 2. Generate the synthetic sample or optional BYOD

The default sample is **synthetic**: a 512 x 512 white canvas drawn in this cell with a filled red square, a filled blue circle, and the text `DIMER 2026` rendered with Pillow's built-in font — so it needs no download, contains no personal data, and is reproducible from code (no randomness, no seed; its pixel SHA-256 is printed and exported). The drawn text is the **known OCR reference**, and the drawn square's rectangle is kept so Section 5 can compare the detector's boxes against it. A drawing is not a photograph, so every output it produces is smoke/sanity evidence that the code path works, not a quality measurement and not benchmark evidence: the model card's smoke labelled a plain red square `flag`.

BYOD is optional and disabled by default. Expected BYOD input: one image file that Pillow can open (PNG, JPEG, WebP, ...), any mode (converted to RGB), with both sides between 1 and `MAX_IMAGE_SIDE` = 4096 px; it will be resized to 768 x 768 regardless of aspect ratio. If your image contains text you know exactly, put it in `OCR_REFERENCE` and Section 5 scores OCR against it; leave it empty otherwise. The upload stays inside this runtime.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image, ImageDraw, ImageFont

USE_BYOD = False  # @param {type:"boolean"}
OCR_REFERENCE = ''  # @param {type:"string"}
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    image_name = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[image_name]))
    image.load()
    sample_kind = 'BYOD upload'
    ocr_reference = OCR_REFERENCE.strip() or None
    drawn_square = None
else:
    # Deterministic drawing: a red square, a blue circle and one line of text in the built-in font.
    image = Image.new('RGB', (512, 512), (255, 255, 255))
    draw = ImageDraw.Draw(image)
    drawn_square = (64, 64, 224, 224)
    draw.rectangle(drawn_square, fill=(220, 30, 30))
    draw.ellipse((300, 96, 460, 256), fill=(30, 60, 220))
    ocr_reference = 'DIMER 2026'
    draw.text((96, 360), ocr_reference, fill=(0, 0, 0), font=ImageFont.load_default(size=48))
    image_name = 'synthetic_shapes_text_512'
    sample_kind = 'synthetic (drawn in this cell)'
sample_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
print({'sample': image_name, 'sample_kind': sample_kind, 'mode': image.mode, 'size': image.size, 'ocr_reference': ocr_reference, 'drawn_square': drawn_square, 'pixel_sha256': sample_sha256})

## 3. Validate the input against the pipeline ceilings and list the task contracts

The pipeline enforces its ceilings through constants imported here from the package so the values shown are the ones in force: `MAX_IMAGE_SIDE` (either side, px), `MAX_NEW_TOKENS` (hard ceiling on `max_new_tokens`; `DEFAULT_MAX_NEW_TOKENS` is the default), `MAX_TEXT_CHARS` (caption text for phrase grounding), and `NUM_BEAMS` (the snapshot's beam count). `TASKS` is the complete set of task tokens the pipeline accepts, split into `TASKS_WITHOUT_TEXT` and `TASKS_WITH_TEXT`; anything else is rejected. This cell surfaces them, prints the per-capability input/output contract for the three tasks demonstrated, and checks the image before any model work; `run()` re-applies the same rules authoritatively and raises `TypeError`/`ValueError` on its own. **What the pipeline changes about your image:** the processor resizes it to exactly 768 x 768 (bicubic, ImageNet mean/std) — nothing is cropped, but a non-square image is distorted — and region outputs are mapped back to your input's pixel coordinates. The notebook itself does not resize, crop, or subsample.

In [ ]:
from florence2_vision_language_pipeline import DEFAULT_MAX_NEW_TOKENS, MAX_IMAGE_SIDE, MAX_NEW_TOKENS, MAX_TEXT_CHARS, NUM_BEAMS, TASKS, TASKS_WITH_TEXT, TASKS_WITHOUT_TEXT

ceilings = {'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'NUM_BEAMS': NUM_BEAMS}
print(ceilings)
print({'TASKS_WITHOUT_TEXT': TASKS_WITHOUT_TEXT, 'TASKS_WITH_TEXT': TASKS_WITH_TEXT})
capabilities = {
    '<CAPTION>': {'input': 'image only', 'output': 'result: str (one short caption); no score', 'max_new_tokens': 64},
    '<OD>': {'input': 'image only', 'output': 'result: {bboxes: [[x1, y1, x2, y2], ...] in input pixels, labels: [str, ...]}; no per-box score', 'max_new_tokens': DEFAULT_MAX_NEW_TOKENS},
    '<OCR>': {'input': 'image only', 'output': 'result: str (transcribed text, reading order chosen by the model); no score', 'max_new_tokens': 128},
}
for task, contract in capabilities.items():
    print(task, contract)
problems = []
width, height = image.size
if width < 1 or height < 1 or max(width, height) > MAX_IMAGE_SIDE:
    problems.append(f'image side {image.size} outside 1..MAX_IMAGE_SIDE={MAX_IMAGE_SIDE} px: resize the image and rerun Section 2')
if any(task not in TASKS for task in capabilities):
    problems.append('a demonstrated task token is not in TASKS')
if any(contract['max_new_tokens'] > MAX_NEW_TOKENS for contract in capabilities.values()):
    problems.append(f'a max_new_tokens setting exceeds MAX_NEW_TOKENS={MAX_NEW_TOKENS}')
if problems:
    raise ValueError('input rejected before model execution: ' + '; '.join(problems))
print({'width': width, 'height': height, 'model_input': '768x768 resize (aspect ratio not preserved)', 'decoding': f'beam search, num_beams={NUM_BEAMS}, do_sample=False', 'within_ceilings': True})

## 4. Stage, verify, and resolve the pinned model

The public API pins the exact upstream model repository and immutable 40-hex revision (`MODEL_ID`/`MODEL_REVISION` are imported from the package, never typed here) and loads with `trust_remote_code=False` through the native `transformers` Florence-2 classes; the loader also refuses a checkpoint whose weights do not map cleanly onto that architecture (`output_loading_info` must report no missing, unexpected or mismatched keys). The repository commits the DIMER snapshot manifest (`weights/florence-2-large-community/dimer-base-manifest.json`: 12 entries with byte sizes and SHA-256 digests) and the small config/processor/tokenizer files, but git-ignores the 1.55 GB `model.safetensors`, so a fresh clone must stage that file first. The package's `stage_missing_files(WEIGHTS_DIR, allow_download=True)` fetches only the manifest-listed files that are absent, from the Hub at the pinned revision, into the repository's weights directory, and returns the list it fetched (`['model.safetensors']` on a fresh clone, `[]` when everything is already staged); it refuses to stage if the committed manifest disagrees with the package's pinned identity. `verify_snapshot()` then re-hashes every listed file and raises on the first size or digest mismatch; only afterwards does `from_pretrained` build processor and model from that verified directory with `local_files_only=True` (`source: local-snapshot`). The effective model identity and the selected device are printed before inference.

In [ ]:
from florence2_vision_language_pipeline import MODEL_ID, MODEL_KEY, MODEL_REVISION, Florence2Pipeline, character_error_rate, stage_missing_files, verify_snapshot

print({'model_id': MODEL_ID, 'revision': MODEL_REVISION})
WEIGHTS_DIR = ROOT / 'weights' / MODEL_KEY
# Only the manifest-listed files that are absent are fetched, at the immutable revision the
# package pins; verify_snapshot then checks every byte count and SHA-256 before anything is loaded.
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'fetched': fetched, 'from': MODEL_ID, 'revision': MODEL_REVISION})
snapshot_info = verify_snapshot(WEIGHTS_DIR)
print({'snapshot_path': snapshot_info['path'], 'files': len(snapshot_info['files']), 'total_bytes': snapshot_info.get('totalBytes')})
pipe = Florence2Pipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': pipe.device, 'source': pipe.source, 'dtype': 'float16' if pipe.device.startswith('cuda') else 'float32'})

## 5. Run the three capabilities and evaluate where a reference exists

Each call to `run(image, task, *, max_new_tokens=..., num_beams=...)` returns `task`, `result` (the task-parsed value), `generated_text` (the raw decoder output with its special tokens), `image_size`, the `generation` settings actually used (`max_new_tokens`, `num_beams`, `do_sample: False`), device, source and model identity. The settings are printed for every call because they change the output: a caption cut off by `max_new_tokens` is a truncated caption, and a different beam count is a different search.

**Capability A — `<CAPTION>`:** input the image; output one short caption string. No score, no metric: caption quality needs human judgement or reference captions plus a caption metric, neither of which the repository ships. **Capability B — `<OD>`:** input the image; output `{bboxes, labels}` in input-pixel coordinates with **no confidence scores** (Florence-2 emits none), so there is no threshold to set and nothing to calibrate; detection quality needs annotated boxes and the caller's own mAP code. The overlap (IoU) between the best box and the drawn square is printed as a **sanity observation** on the default sample, not a metric. **Capability C — `<OCR>`:** input the image; output the transcribed text as one string. The repository ships `character_error_rate(reference, hypothesis)` — character-level Levenshtein distance divided by the reference length — and it is computed **only** when a reference is known: on the default sample the reference is the text the notebook drew, so the figure is a sanity check of the code path on a rendered font, not an OCR benchmark; on a BYOD image it needs `OCR_REFERENCE`. No baseline is reported for any capability (none is meaningful without labelled data). Outputs are fluent even when wrong — the model can name objects that are not there or transpose characters — and nothing in the output signals it. The runtime figures are measured on the runtime identified in Section 1 for this one image and include the first-call warm-up.

In [ ]:
import time

results = {}
timings = {}
for task, contract in capabilities.items():
    started = time.perf_counter()
    results[task] = pipe.run(image, task, max_new_tokens=contract['max_new_tokens'], num_beams=NUM_BEAMS)
    timings[task] = round(time.perf_counter() - started, 3)
    print({'task': task, 'seconds': timings[task], 'generation': results[task]['generation'], 'result': results[task]['result']})
caption = results['<CAPTION>']['result']
detections = results['<OD>']['result']
ocr_text = results['<OCR>']['result']
checks = {
    'caption_is_text': isinstance(caption, str) and bool(caption.strip()),
    'od_boxes_and_labels_aligned': isinstance(detections, dict) and len(detections.get('bboxes', [])) == len(detections.get('labels', [])),
    'od_boxes_inside_image': all(0 <= x1 <= x2 <= image.width and 0 <= y1 <= y2 <= image.height for x1, y1, x2, y2 in detections.get('bboxes', [])),
    'ocr_is_text': isinstance(ocr_text, str),
    'deterministic_settings': all(r['generation']['do_sample'] is False and r['generation']['num_beams'] == NUM_BEAMS for r in results.values()),
}
if not all(checks.values()):
    raise RuntimeError(f'capability output failed a sanity check: {checks}')
print({'checks': checks})


def _iou(a, b):
    ix1, iy1, ix2, iy2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union else 0.0


observations = {}
if drawn_square is not None and detections.get('bboxes'):
    best = max(detections['bboxes'], key=lambda box: _iou(box, drawn_square))
    observations['best_box_iou_with_drawn_square'] = round(_iou(best, drawn_square), 3)
    print({'observation': observations, 'note': 'sanity observation on a drawing; not a detection metric'})
metrics = {}
if ocr_reference is not None:
    metrics['character_error_rate'] = character_error_rate(ocr_reference, ocr_text.strip())
    print({'ocr_reference': ocr_reference, 'ocr_hypothesis': ocr_text.strip(), 'sample_metrics': metrics, 'estimation': 'single image against known drawn/supplied text; sanity evidence only'})
else:
    print('no metric is reported for OCR: no reference text was supplied; captions and detections have no metric in this repository')

## 6. Preview the detections

The preview draws the `<OD>` boxes and labels onto a copy of the input with Pillow so you can see where the detector placed them; it is a visual aid only — the machine-readable boxes exported in the next section are the outputs intended for downstream use. On the synthetic drawing expect boxes around the shapes with whatever labels the model chose; the model card's smoke called a red square `flag`.

In [ ]:
preview = image.convert('RGB').copy()
draw_preview = ImageDraw.Draw(preview)
for (x1, y1, x2, y2), label in zip(detections.get('bboxes', []), detections.get('labels', [])):
    draw_preview.rectangle((x1, y1, x2, y2), outline=(0, 160, 0), width=3)
    draw_preview.text((x1 + 4, y1 + 4), label, fill=(0, 160, 0))
try:
    from IPython.display import display
    display(preview)
except ImportError:
    print({'preview': 'IPython display unavailable; the preview PNG is written in the next section'})

## 7. Export results and provenance

Two files are written under `outputs/`: the detection preview PNG, and a JSON record with one entry per capability (task token, parsed `result`, raw `generated_text`, generation settings, seconds), the sanity checks and the IoU observation, the OCR metric block (empty without a reference), the ceilings in force, the sample identity (name, kind, size, pixel digest, OCR reference, drawn square), the repository revision, the model identifier and immutable revision, the verified snapshot summary, and the runtime identity (Python, `torch`, `transformers`, Pillow, device, dtype). No credentials are involved in any step, so none can reach the export.

In [ ]:
import json

os.makedirs('outputs', exist_ok=True)
preview.save('outputs/florence2_vision_language_preview.png')
payload = {
    'capabilities': {
        task: {'result': r['result'], 'generated_text': r['generated_text'], 'generation': r['generation'], 'seconds': timings[task]}
        for task, r in results.items()
    },
    'sanity_checks': checks,
    'observations': observations,
    'metrics': metrics,
    'ceilings': ceilings,
    'tasks_exposed': list(TASKS),
    'sample': {'name': image_name, 'kind': sample_kind, 'width': image.width, 'height': image.height, 'pixel_sha256': sample_sha256, 'ocr_reference': ocr_reference, 'drawn_square': drawn_square},
    'preview_file': 'outputs/florence2_vision_language_preview.png',
    'repository_revision': REPO_SHA,
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'snapshot': {'path': snapshot_info['path'], 'files': len(snapshot_info['files']), 'total_bytes': snapshot_info.get('totalBytes')},
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'pillow': PIL.__version__,
        'device': pipe.device,
        'dtype': 'float16' if pipe.device.startswith('cuda') else 'float32',
    },
}
with open('outputs/florence2_vision_language_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))
print('outputs/florence2_vision_language_result.json')

## Interpretation and limits

The three outputs are generated text parsed per task: a caption string, boxes with labels and **no confidence scores**, and an OCR string. None of them carries a probability, and the only quantity the notebook scores is OCR against text it knows — on the default sample the text it drew itself, which makes the character error rate a code-path sanity check on a rendered font, not an OCR benchmark; captions and detections have no metric in this repository and need human judgement or annotated references plus the caller's own evaluation code. Every output is fluent whether or not it is right: the model can describe objects that are not there, label a plain square as a `flag`, or transpose characters, and nothing in the output signals it. The image is squashed to 768 x 768 before encoding, so thin or off-aspect content is distorted; region outputs are mapped back to input pixels. Decoding is deterministic beam search (3 beams, no sampling) on a fixed device and dtype; CPU float32 and CUDA float16 can produce different text. The repository exposes the nine task tokens in `TASKS` — including phrase grounding, which takes a caption as `text_input` — and nothing else: no segmentation, no region-to-category or region-to-description prompts, no open-vocabulary detection, no VQA, no batching.

Successful execution proves that the recorded repository revision can bootstrap in a fresh runtime, stage and digest-verify the pinned model snapshot, validate the demonstrated input against the enforced ceilings, execute the public pipeline path for three task tokens with explicit generation settings, and emit the shown machine-readable outputs in the tested runtime. It does **not** establish benchmark superiority, caption, detection or OCR quality on any domain, safety for high-consequence decisions, or production fitness on an unseen domain.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 4: a staged file is incomplete or altered — delete it from `weights/florence-2-large-community/` and rerun Section 4. `RuntimeError: checkpoint does not match the native Florence-2 architecture` in Section 4: the staged weights are not the pinned converted checkpoint — re-stage. A `ValueError` naming `MAX_IMAGE_SIDE` in Section 3: resize the BYOD image and rerun from Section 2. A truncated caption or OCR string: raise that task's `max_new_tokens` in Section 3 (up to `MAX_NEW_TOKENS`). A "slow image processor" notice from `transformers` is expected and harmless.

**Next experiments.** Upload a photograph with readable signage and supply `OCR_REFERENCE` to see how the character error rate behaves on real text; run `<CAPTION_TO_PHRASE_GROUNDING>` with the caption from Capability A as `text_input` and compare its boxes with `<OD>`; run the same image on a CUDA runtime and diff the float16 outputs against the CPU float32 ones. None of these turns the sample result into evidence of production fitness.

## References

- Repository README: `../README.md`
- Repository model card: `../MODEL_CARD.md`
- Weight provenance and pin history: `../docs/WEIGHTS.md`
- Pinned converted checkpoint: https://huggingface.co/florence-community/Florence-2-large
- Original weights and licence: https://huggingface.co/microsoft/Florence-2-large
- Florence-2 paper: https://arxiv.org/abs/2311.06242
- Transformers Florence-2 documentation: https://huggingface.co/docs/transformers/model_doc/florence2